# **LangChain**


Estimated time needed: **40** minutes


## Overview


LangChain is an open-source framework uniquely designed to empower the development of applications leveraging large language models (LLMs). It stands out by providing essential tools and abstractions that enhance the customization, accuracy, and relevance of the information generated by these models.

At its core, LangChain offers a generic interface compatible with nearly any LLM. This facilitates a centralized development environment where data scientists can seamlessly integrate LLM applications with various external data sources and software workflows. This integration is crucial for those looking to harness the full potential of AI in their processes.

One of the most powerful features of LangChain is its module-based approach. This approach allows flexibility in performing experiments and optimizations of interactions with LLMs. Data scientists can dynamically compare prompts and switch between foundation models without significant code modifications. This saves valuable development time and enhances the ability to fine-tune applications to meet specific needs.


<figure>
    <img src="images/01 - langchain.jpg" width="50%" alt="langchain">
    <figcaption><a href="https://navan.ai/blog/what-is-langchain/">source</a></figcaption>
</figure>


By participating in this lab, you will dive into how LangChain simplifies the complex process of integrating advanced AI capabilities into practical applications. You will learn the core concepts of LangChain and how to use Langchain's innovative features to build more intelligent, responsive, and efficient applications. Whether you are a developer, a data scientist, or an AI enthusiast, this lab will equip you with a deep understanding of how to leverage LangChain for crafting cutting-edge AI solutions.


## __Table of Contents__

<ol>
    <li><a href="#Overview">Overview</a></li>
    <li><a href="#Objectives">Objectives</a></li>
    <li>
        <a href="#Setup">Setup</a>
        <ol>
            <li><a href="#Installing-required-libraries">Installing required libraries</a></li>
            <li><a href="#Importing-required-libraries">Importing required libraries</a></li>
        </ol>
    </li>
    <li>
        <a href="#LangChain-concepts">LangChain concepts</a>
        <ol>
            <li><a href="#Model">Model</a></li>
            <li><a href="#Chat-model">Chat model</a></li>
            <li><a href="#Chat-message">Chat message</a></li>
            <li><a href="#Prompt-templates">Prompt templates</a></li>
            <li><a href="#Example-selectors">Example selectors</a></li>
            <li><a href="#Output-parsers">Output parsers</a></li>
            <li><a href="#Documents">Documents</a></li>
            <li><a href="#Memory">Memory</a></li>
            <li><a href="#Chains">Chains</a></li>
            <li><a href="#Agents">Agents</a></li>
        </ol>
    </li>
</ol>

<a href="#Exercises">Exercises</a>
<ol>
    <li><a href="#Exercise-1:-Try-with-another-LLM">Exercise 1: Try with another LLM</a></li>
    <li><a href="#Exercise-2:-Split-the-document-with-another-separator">Exercise 2: Split the document with another separator</a></li>
    <li><a href="#Exercise-3:-Create-an-agent-to-talk-with-CSV-data">Exercise 3: Create an agent to talk with CSV data</a></li>
</ol>


## Objectives

After completing this lab, you will be able to:

- Grasp the core features of Langchain, including prompt templates, chains, and agents, emphasizing its role in enhancing LLM customization and output relevance. **(Framework understanding)**:

- Explore LangChain's modular flexibility, which allows for dynamic adjustments to prompts and models without extensive code changes. **(Modular approach)**

- Discover how to enhance LLM applications by integrating retrieval-augmented generation (RAG) techniques with LangChain. This enables more accurate and context-aware responses by leveraging external data sources. **(Retrieval-augmented integration)**


## Setup


For this lab, you will be using the following libraries:

*   [`ibm-watson-ai`, `ibm-watson-machine-learning`](https://ibm.github.io/watson-machine-learning-sdk/index.html) for using LLMs from IBM's watsonx.ai.
*   [`langchain`, `langchain-ibm`, `langchain-community`, `langchain-experimental`](https://www.langchain.com/) for using relevant features from LangChain.
*   [`pypdf`](https://pypi.org/project/pypdf/) is an open-source pure-python PDF library capable of splitting, merging, cropping, and transforming the pages of PDF files.
*   [`chromadb`](https://www.trychroma.com/) is an open-source vector database used to store embeddings.


### Installing required libraries

The following required libraries are __not__ pre-installed in the Skills Network Labs environment. __You must run the following cell__ to install them:

**Note:** The version has been specified here to pin it. It's recommended that you do the same. Even if the library is updated in the future, the installed version will still support this lab work.

The installation might take approximately 2-3 minutes. 

Since `%%capture` is being used to capture the installation process, you won't see the output. However, once the installation is complete, you will see a number beside the cell.


In [1]:
# %%capture
# !pip install --force-reinstall --no-cache-dir tenacity --user
# !pip install "ibm-watsonx-ai==1.0.4" --user
# !pip install "ibm-watson-machine-learning==1.0.357" --user
# !pip install "langchain==0.2.1" --user
# !pip install "langchain-community==0.2.1" --user
# !pip install "langchain-ibm==0.1.7" --user
# !pip install "langchain-experimental==0.0.59" --user
# !pip install "langchainhub==0.1.17" --user
# !pip install "pypdf==4.2.0" --user
# !pip install "chromadb==0.4.24" --user
# !pip install "posthog==3.5.0" --user


After you install the libraries, restart your kernel. You can do that by clicking the **Restart the kernel** icon as shown in the screenshot below:

<img src="images/02 - restart-kernel.jpg" style="margin:1cm;width:90%;border:1px solid grey" alt="Restart kernel">


### Importing required libraries

The following imports the required libraries:


In [2]:
# You can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import ModelTypes
from ibm_watsonx_ai.foundation_models.extensions.langchain import WatsonxLLM

## LangChain concepts


### Model


A large language model (LLM) serves as the interface for the AI's capabilities. It processes plain text input and generates text output, forming the core functionality needed to complete various tasks. When integrated with LangChain, it becomes a powerful tool, providing the foundational structure necessary for building and deploying sophisticated AI applications.


The following will construct a `mistralai/mistral-small-3-1-24b-instruct-2503` watsonx.ai inference model object:


In [3]:
from langchain_core.language_models.llms import LLM
from langchain_core.callbacks.manager import CallbackManagerForLLMRun
from typing import Any, List, Optional, Mapping
from openai import OpenAI
import os

os.environ["HF_TOKEN"] = ""   # keep the one you already have

# -------------------------------------------------
# 1. Create the OpenAI-compatible client (once)
# -------------------------------------------------
client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_TOKEN"],          # or HUGGINGFACEHUB_API_TOKEN
)

# -------------------------------------------------
# 2. Proper LangChain LLM wrapper
# -------------------------------------------------
class HuggingFaceRouterLLM(LLM):
    """Thin LangChain wrapper around Hugging Face Inference Providers (router)."""

    model_id: str = "Qwen/Qwen2.5-7B-Instruct"
    max_tokens: int = 256
    temperature: float = 0.5

    @property
    def _llm_type(self) -> str:
        return "huggingface_router"

    def _call(
        self,
        prompt: str,
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> str:
        response = client.chat.completions.create(
            model=self.model_id,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=self.max_tokens,
            temperature=self.temperature,
            stop=stop,
        )
        return response.choices[0].message.content

    @property
    def _identifying_params(self) -> Mapping[str, Any]:
        return {
            "model_id": self.model_id,
            "max_tokens": self.max_tokens,
            "temperature": self.temperature,
        }


In [32]:
# model_id = 'meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8'
model_id = 'Qwen/Qwen3.8-27B'
# -------------------------------------------------
# 3. Instantiate and use it
# -------------------------------------------------
model = HuggingFaceRouterLLM(
    model_id=model_id,   # change to any model that worked for you
    max_tokens=256,
    temperature=0.5,
)


Let's use a simple example to let the model generate some text:


In [33]:
msg = model.generate(["In today's sales meeting, we "])
print(msg.generations[0][0].text)



It looks like your sentence is incomplete. Did you mean to say something like:

- “In today’s sales meeting, we **discussed the new pricing strategy**.”
- “In today’s sales meeting, we **reviewed last month’s performance**.”
- “In today’s sales meeting, we **presented the quarterly results**.”

If you’d like, send the rest of the sentence and I can help you complete or polish it.


### Chat model


Chat models support the assignment of distinct roles to conversation messages, helping to distinguish messages from the AI, users, and instructions such as system messages.


In [48]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage, SystemMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.callbacks.manager import CallbackManagerForLLMRun
from typing import Any, List, Optional, Mapping
from openai import OpenAI
import os

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACEHUB_API_TOKEN"),
)

class HuggingFaceRouterChat(BaseChatModel):
    """Proper ChatModel wrapper for Hugging Face router."""

    model_id: str = "Qwen/Qwen2.5-7B-Instruct"
    max_tokens: int = 256
    temperature: float = 0.5

    @property
    def _llm_type(self) -> str:
        return "huggingface_router_chat"

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> ChatResult:
        # Convert LangChain messages → OpenAI format
        openai_messages = []
        for m in messages:
            if isinstance(m, SystemMessage):
                role = "system"
            elif isinstance(m, HumanMessage):
                role = "user"
            elif isinstance(m, AIMessage):
                role = "assistant"
            else:
                role = "user"
            openai_messages.append({"role": role, "content": m.content})

        response = client.chat.completions.create(
            model=self.model_id,
            messages=openai_messages,
            max_tokens=self.max_tokens,
            temperature=self.temperature,
            stop=stop,
        )

        content = response.choices[0].message.content
        generation = ChatGeneration(message=AIMessage(content=content))
        return ChatResult(generations=[generation])

    @property
    def _identifying_params(self) -> Mapping[str, Any]:
        return {
            "model_id": self.model_id,
            "max_tokens": self.max_tokens,
            "temperature": self.temperature,
        }

# Factory
def chat_model(model_id: str, max_tokens: int = 256, temperature: float = 0.5):
    return HuggingFaceRouterChat(
        model_id=model_id,
        max_tokens=max_tokens,
        temperature=temperature,
    )

llm_chat = chat_model(model_id=model_id)

In [49]:
print(llm_chat.invoke("Who is man's best friend?"))

content='\n\nThe dog is traditionally called “man’s best friend.”' id='run-4d3bc558-bbbf-4f2a-b24e-608bd1070016-0'


### Chat message


The chat model takes a list of messages as input and returns a message. All messages have a role and a content property. There are a few different types of messages. The most commonly used are the following:
- `SystemMessage`: Used for priming AI behavior, usually passed in as the first in a sequence of input messages.
- `HumanMessage`: Represents a message from a person interacting with the chat model.
- `AIMessage`: Represents a message from the chat model. This can be either text or a request to invoke a tool.

More messages types can be found at [https://docs.langchain.com/oss/python/langchain/messages](https://docs.langchain.com/oss/python/langchain/messages).


The following imports the most common message type classes from LangChain:


In [50]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

Now let's create a few messages that simulate a chat experience with the bot:


In [51]:
msg = llm_chat.invoke(
    [
        SystemMessage(content="You are a helpful AI bot that assists a user in choosing the perfect book to read in one short sentence"),
        HumanMessage(content="I enjoy mystery novels, what should I read?")
    ]
)

In [52]:
print(msg)

content='\n\nTry *The Hound of the Baskervilles* by Arthur Conan Doyle for a classic, atmospheric mystery.' id='run-01edd3ab-1644-438d-a2d9-09e5467a6c40-0'


Notice that the model responded with an `AI` message.


You can use these message types to pass an entire chat history along with the AI's responses to the model:


In [53]:
msg = llm_chat.invoke(
    [
        SystemMessage(content="You are a supportive AI bot that suggests fitness activities to a user in one short sentence"),
        HumanMessage(content="I like high-intensity workouts, what should I do?"),
        AIMessage(content="You should try a CrossFit class"),
        HumanMessage(content="How often should I attend?")
    ]
)

In [54]:
print(msg)

content='\n\nAim for three to four sessions a week with rest days in between to let your body recover and avoid burnout!' id='run-7e45bbd9-e8e4-46dd-8a9c-9549693c76fa-0'


You can also exclude the system message if you want:


In [55]:
msg = llm_chat.invoke(
    [
        HumanMessage(content="What month follows June?")
    ]
)

In [56]:
print(msg)

content='\n\nJuly follows June.' id='run-f7f99916-c357-47d9-a161-1ad60e9be249-0'


### Prompt templates


Prompt templates help translate user input and parameters into instructions for a language model. They can be used to guide a model's response, helping it understand the context and generate relevant and coherent language-based output.

There are several different types of prompt templates.


#### String prompt templates


These prompt templates are used to format a single string, and are generally used for simpler inputs.


In [57]:
from langchain_core.prompts import PromptTemplate

In [58]:
prompt = PromptTemplate.from_template("Tell me one {adjective} joke about {topic}")
input_ = {"adjective": "funny", "topic": "cats"}  # create a dictionary to store the corresponding input to placeholders in prompt template

In [59]:
prompt.invoke(input_)

StringPromptValue(text='Tell me one funny joke about cats')

Note how the prompt was formatted.


#### Chat prompt templates


These prompt templates are used to format a list of messages. These "templates" consist of a list of templates themselves.


In [60]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("user", "Tell me a joke about {topic}")
])

input_ = {"topic": "cats"}

prompt.invoke(input_)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant'), HumanMessage(content='Tell me a joke about cats')])

#### Messages place holder


This prompt template is responsible for adding a list of messages in a particular place. In the above ChatPromptTemplate, you saw how two messages can be formatted, each one a string. But what if you want the user to pass in a list of messages that you would slot into a particular spot? This is how you use MessagesPlaceholder.


In [61]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder("msgs")
])

input_ = {"msgs": [HumanMessage(content="What is the day after Tuesday?")]}

prompt.invoke(input_)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant'), HumanMessage(content='What is the day after Tuesday?')])

You could wrap the prompt and the chat model and pass them into a chain, which could invoke the message.


In [62]:
chain = prompt | llm_chat
response = chain.invoke(input = input_)
print(response)

content='\n\nThe day after Tuesday is Wednesday.' id='run-5de8bddb-21d2-46a0-991c-3e78f70c92da-0'


### Example selectors


If you have a large number of examples, you may need to select which ones to include in the prompt. The Example Selector is the class responsible for doing so.


Example selector types could based on:
- `Similarity`: Uses semantic similarity between inputs and examples to decide which examples to choose.
- `MMR`: Uses Max Marginal Relevance between inputs and examples to decide which examples to choose.
- `Length`: Selects examples based on how many can fit within a certain length
- `Ngram`: Uses ngram overlap between inputs and examples to decide which examples to choose.

Here, you can use the example selector based on length as an example. For more details on other types, please refer to [https://reference.langchain.com/python/langchain-core/example_selectors](https://reference.langchain.com/python/langchain-core/example_selectors).


In [63]:
from langchain_core.example_selectors import LengthBasedExampleSelector
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

# Examples of a pretend task of creating antonyms.
examples = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
    {"input": "energetic", "output": "lethargic"},
    {"input": "sunny", "output": "gloomy"},
    {"input": "windy", "output": "calm"},
]

example_prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="Input: {input}\nOutput: {output}",
)
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    max_length=25,  # The maximum length that the formatted examples should be.
)
dynamic_prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix="Give the antonym of every input",
    suffix="Input: {adjective}\nOutput: ",
    input_variables=["adjective"],
)

In [64]:
example_selector

LengthBasedExampleSelector(examples=[{'input': 'happy', 'output': 'sad'}, {'input': 'tall', 'output': 'short'}, {'input': 'energetic', 'output': 'lethargic'}, {'input': 'sunny', 'output': 'gloomy'}, {'input': 'windy', 'output': 'calm'}], example_prompt=PromptTemplate(input_variables=['input', 'output'], template='Input: {input}\nOutput: {output}'), get_text_length=<function _get_length_based at 0x000001B3B2C42560>, max_length=25, example_text_lengths=[4, 4, 4, 4, 4])

An example with small input, so it selects all examples.


In [65]:
print(dynamic_prompt.format(adjective="big"))

Give the antonym of every input

Input: happy
Output: sad

Input: tall
Output: short

Input: energetic
Output: lethargic

Input: sunny
Output: gloomy

Input: windy
Output: calm

Input: big
Output:


An example with long input, so it selects only one example.


In [66]:
long_string = "big and huge and massive and large and gigantic and tall and much much much much much bigger than everything else"
print(dynamic_prompt.format(adjective=long_string))

Give the antonym of every input

Input: happy
Output: sad

Input: big and huge and massive and large and gigantic and tall and much much much much much bigger than everything else
Output:


### Output parsers


Output parsers are responsible for taking the output of an LLM and transforming it to a more suitable format. This is very useful when you are using LLMs to generate any form of structured data, or to normalize output from chat models and LLMs.


LangChain has lots of different types of output parsers. This is a [list](https://python.langchain.com/v0.2/docs/concepts/#output-parsers) of output parsers LangChain supports. In this lab, you will use the following two output parsers as examples:

- `JSON`: Returns a JSON object as specified. You can specify a Pydantic model and it will return JSON for that model. Probably the most reliable output parser for getting structured data that does NOT use function calling.
- `CSV`: Returns a list of comma separated values.


#### JSON parser


This output parser allows users to specify an arbitrary JSON schema and query LLMs for outputs that conform to that schema.


In [67]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field

In [68]:
# Define your desired data structure.
class Joke(BaseModel):
    setup: str = Field(description="question to set up a joke")
    punchline: str = Field(description="answer to resolve the joke")

In [69]:
# And a query intented to prompt a language model to populate the data structure.
joke_query = "Tell me a joke."

# Set up a parser + inject instructions into the prompt template.
output_parser = JsonOutputParser(pydantic_object=Joke)

format_instructions = output_parser.get_format_instructions()
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": format_instructions},
)

chain = prompt | llm_chat | output_parser

chain.invoke({"query": joke_query})

{'setup': 'Why did the scarecrow win an award?',
 'punchline': 'Because he was outstanding in his field.'}

#### Comma separated list parser


This output parser can be used when you want to return a list of comma-separated items.


In [70]:
from langchain.output_parsers import CommaSeparatedListOutputParser

output_parser = CommaSeparatedListOutputParser()

format_instructions = output_parser.get_format_instructions()
prompt = PromptTemplate(
    template="Answer the user query. {format_instructions}\nList five {subject}.",
    input_variables=["subject"],
    partial_variables={"format_instructions": format_instructions},
)

chain = prompt | granite_llm | output_parser

In [71]:
chain.invoke({"subject": "ice cream flavors"})

['vanilla',
 'chocolate',
 'strawberry',
 'mint chocolate chip',
 'cookies and cream']

### Documents


#### Document object


A `Document` object in `LangChain` contains information about some data. It has two attributes:

- `page_content`: *`str`*: This attribute holds the content of the document\.
- `metadata`: *`dict`*: This attribute contains arbitrary metadata associated with the document. It can be used to track various details such as the document id, file name, and so on.


Let's use an example to illustrate how to create a `Document` object. This is the object type that `LangChain` utilizes for handling text or documents


In [72]:
from langchain_core.documents import Document

In [73]:
Document(page_content="""Python is an interpreted high-level general-purpose programming language. 
                        Python's design philosophy emphasizes code readability with its notable use of significant indentation.""",
         metadata={
             'my_document_id' : 234234,
             'my_document_source' : "About Python",
             'my_document_create_time' : 1680013019
         })

Document(metadata={'my_document_id': 234234, 'my_document_source': 'About Python', 'my_document_create_time': 1680013019}, page_content="Python is an interpreted high-level general-purpose programming language. \n                        Python's design philosophy emphasizes code readability with its notable use of significant indentation.")

Note that you don't have to include metadata if you don't want to:


In [74]:
Document(page_content="""Python is an interpreted high-level general-purpose programming language. 
                        Python's design philosophy emphasizes code readability with its notable use of significant indentation.""")

Document(page_content="Python is an interpreted high-level general-purpose programming language. \n                        Python's design philosophy emphasizes code readability with its notable use of significant indentation.")

#### Document loaders


Document loaders in LangChain are designed to load documents from a variety of sources. For instance, if you wish to load a PDF paper and have it read by LLM using LangChain.

LangChain offers over 100 distinct document loaders, along with integrations with other major providers in this field, such as AirByte and Unstructured. These integrations enable the loading of all kinds of documents (HTML, PDF, code) from various locations (private S3 buckets, public websites).

You can find a list of document types that LangChain can load at [https://python.langchain.com/v0.1/docs/integrations/document_loaders/](https://python.langchain.com/v0.1/docs/integrations/document_loaders/).

In this lab, you will be using the PDF loader and the URL/Website loader as examples.


##### PDF loader


By using the PDF loader, you can load a PDF file as a `Document` object.

In this case, you are loading a paper about LangChain. You can access and read the paper at [https://doi.org/10.48550/arXiv.2403.05568](https://doi.org/10.48550/arXiv.2403.05568).


In [75]:
from langchain_community.document_loaders import PyPDFLoader

In [76]:
# loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")
loader = PyPDFLoader("assets/langchain-paper.pdf")


In [77]:
document = loader.load()

Here, `document` is a `Document` object with `page_content` and `metadata`:


In [78]:
document[2]  # take a look at the page 2

Document(metadata={'source': 'assets/langchain-paper.pdf', 'page': 2}, page_content=' \nFigure 2. An AIMessage illustration \nC. Prompt Template \nPrompt templates [10] allow you to structure input for LLMs. \nThey provide a convenient way to format user inputs and \nprovide instructions to generate responses. Prompt templates \nhelp ensure that the LLM understands the desired context and \nproduces relevant outputs. \nThe prompt template classes in LangChain are built to \nmake constructing prompts with dynamic inputs easier. Of \nthese classes, the simplest is the PromptTemplate. \nD. Chain \nChains [11] in LangChain refer to the combination of \nmultiple components to achieve specific tasks. They provide \na structured and modular approach to building language \nmodel applications. By combining different components, you \ncan create chains that address various u se cases and \nrequirements. Here are some advantages of using chains: \n• Modularity: Chains allow you to break down \nco

In [79]:
print(document[1].page_content[:1000])  # print the page 1's first 1000 tokens

LangChain helps us to unlock the ability to harness the 
LLM’s immense potential in tasks such as document analysis, 
chatbot development, code analysis, and countless other 
applications. Whether your desire is to unlock deeper natural 
language understanding , enhance data, or circumvent 
language barriers through translation, LangChain is ready to 
provide the tools and programming support you need to do 
without it that it is not only difficult but also fresh for you. Its 
core functionalities encompass: 
1. Context-Aware Capabilities: LangChain facilitates the 
development of applications that are inherently 
context-aware. This means that these applications can 
connect to a language model and draw from various 
sources of context, such as prompt instructions, a few-
shot examples, or existing content, to ground their 
responses effectively. 
2. Reasoning Abilities: LangChain equips applications 
with the capacity to reason effectively. By relying on a 
language model, these appl

##### URL and website loader


You can also load content from a URL or website into a `Document` object:


In [80]:
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [81]:
loader = WebBaseLoader("https://python.langchain.com/v0.2/docs/introduction/")

In [82]:
web_data = loader.load()

In [83]:
print(web_data[0].page_content[:1000])

LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to London on October 13! Get ticketsDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryEvent streamingStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatternsIntegrationsAdvanced usageGuardrailsRuntimeContext engineeringMCPHuman-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIProductionDeploymentObservabilityOn this page Create an agent Core benefitsLangChain overviewCopy pageCopy pageLangChain 

#### Text splitters


Once you've loaded documents, you'll often want to transform them to better suit your application.

The simplest example is you may want to split a long document into smaller chunks that can fit into your model's context window. LangChain has a number of built-in document transformers that make it easy to split, combine, filter, and otherwise manipulate documents.


At a high level, text splitters work as follows:

1. Split the text up into small, semantically meaningful chunks (often sentences).
2. Start combining these small chunks into a larger chunk until you reach a certain size (as measured by some function).
3. Once you reach that size, make that chunk its own piece of text and then start creating a new chunk of text with some overlap (to keep context between chunks).

[Here](https://python.langchain.com/v0.1/docs/modules/data_connection/document_transformers/) is a list of types of text splitters LangChain support.


Let's use a simple `CharacterTextSplitter` as an example to split the langchain paper you just loaded.

This is the simplest method. This splits based on characters (by default "\n\n") and measures chunk length by number of characters.


In [84]:
from langchain.text_splitter import CharacterTextSplitter

In [85]:
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="\n")  # define chunk_size which is length of characters, and also separator.
chunks = text_splitter.split_documents(document)
print(len(chunks))

147


It splits the document into 148 chunks. Let's look at the content of a chunk:


In [86]:
chunks[5].page_content   # take a look at any chunk's page content

'individuals seeking guidance and support in these critical areas. \nMindGuide lever ages the capabilities of LangChain and its \nChatModels, specifically Chat OpenAI, as the bedrock of its'

#### Embedding models


Embedding models are specifically designed to interface with text embeddings. 

Embeddings generate a vector representation for a given piece of text. This is advantageous as it allows you to conceptualize text within a vector space. Consequently, you can perform operations such as semantic search, where you identify pieces of text that are most similar within the vector space.


There are lots of embedding model providers (OpenAI, IBM, Hugging Face, etc.). Here, you'll use the embedding model from huggingface to deal with the text.


The following embeds content in each of the chunks. You can then output the first 5 numbers in the vector representation of the content of the first chunk:


In [87]:
from langchain_huggingface import HuggingFaceEmbeddings
def huggingface_embedding():
    # We replace the IBM Slate model with MiniLM, a highly optimized, 
    # widely used open-source embedding model that performs identically for RAG.
    
    hf_embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={'device': 'cuda'},  # This routes the math to your NVIDIA GTX 1650
        encode_kwargs={'normalize_embeddings': True} # Standard best practice for vector similarity
    )
    
    return hf_embeddings

hf_embedding = huggingface_embedding()

In [88]:
texts = [text.page_content for text in chunks]

embedding_result = hf_embedding.embed_documents(texts)
embedding_result[0][:5]

[0.0008826503762975335,
 0.013385613448917866,
 0.020798642188310623,
 -0.008458823896944523,
 -0.06765623390674591]

#### Vector stores


One of the most common ways to store and search over unstructured data is to embed it and store the resulting embedding vectors, and then at query time to embed the unstructured query and retrieve the embedding vectors that are 'most similar' to the embedded query. A [vector store](https://python.langchain.com/v0.1/docs/modules/data_connection/vectorstores/) takes care of storing embedded data and performing vector search for you.


There are many great vector store options, here `Chroma` as an example is being used.


In [89]:
from langchain.vectorstores import Chroma

You have the embedding model perform the embedding process and store the resulting vectors in the Chroma vector database.


In [90]:
docsearch = Chroma.from_documents(chunks, hf_embedding)

Then, you could use a similarity search strategy to retrieve the information that is related to the query you set.

The model will return a list of similar/relevant document chunks. Here, you can print the contents of the most similar chunk:


In [91]:
query = "Langchain"
docs = docsearch.similarity_search(query)
print(docs[0].page_content)

II. LANGCHAIN 
LangChain, with its open -source essence, emerges as a 
promising solution, aiming to simplify the complex process of 
developing applications powered by large language models


#### Retrievers


A retriever is an interface that returns documents given an unstructured query. It is more general than a vector store. A retriever does not need to be able to store documents, only to return (or retrieve) them. Vector stores can be used as the backbone of a retriever, but there are other types of retrievers as well.

Retrievers accept a string `query` as input and return a list of `Document`'s as output.


A list of advanced retrieval types LangChain could support is available at [https://docs.langchain.com/oss/python/integrations/retrievers](https://docs.langchain.com/oss/python/integrations/retrievers). Let's introduce the `Vector store-backed retriever` and `Parent document retriever` as examples.


##### Vector store-backed retriever


A vector store retriever is a retriever that uses a vector store to retrieve documents. It is a lightweight wrapper around the vector store class to make it conform to the retriever interface. It uses the search methods implemented by a vector store, like similarity search and MMR (Maximum marginal relevance), to query the texts in the vector store.

Since we've constructed a vector store `docsearch`, it's very easy to construct a retriever.


In [92]:
retriever = docsearch.as_retriever()

In [93]:
docs = retriever.invoke("Langchain")

In [94]:
docs[0]

Document(metadata={'page': 0, 'source': 'assets/langchain-paper.pdf'}, page_content='II. LANGCHAIN \nLangChain, with its open -source essence, emerges as a \npromising solution, aiming to simplify the complex process of \ndeveloping applications powered by large language models')

Note that the results are identical to the ones obtained using the similarity search strategy.


##### Parent document retriever


When splitting documents for retrieval, there are often conflicting desires:

1. You may want small documents so their embeddings can most accurately reflect their meaning. If too long, then the embeddings can lose meaning.
2. You want to have long enough documents to retain the context of each chunk.

The `ParentDocumentRetriever` strikes that balance by splitting and storing small chunks of data. During retrieval, it first fetches the small chunks but then looks up the parent IDs for them and returns those larger documents.


In [95]:
from langchain.retrievers import ParentDocumentRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.storage import InMemoryStore

In [97]:
# Set two splitters. One is with big chunk size (parent) and one is with small chunk size (child)
parent_splitter = CharacterTextSplitter(chunk_size=2000, chunk_overlap=20, separator='\n')
child_splitter = CharacterTextSplitter(chunk_size=400, chunk_overlap=20, separator='\n')

vectorstore = Chroma(
    collection_name="split_parents", embedding_function=hf_embedding
)

# The storage layer for the parent documents
store = InMemoryStore()

In [98]:
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

In [99]:
retriever.add_documents(document)

These are a number of large chunks.


In [100]:
len(list(store.yield_keys()))

16

Let's make sure the underlying vector store still retrieves the small chunks.


In [101]:
sub_docs = vectorstore.similarity_search("Langchain")

In [102]:
print(sub_docs[0].page_content)

LangChain helps us to unlock the ability to harness the 
LLM’s immense potential in tasks such as document analysis, 
chatbot development, code analysis, and countless other 
applications. Whether your desire is to unlock deeper natural 
language understanding , enhance data, or circumvent 
language barriers through translation, LangChain is ready to


And then retrieve the relevant large chunk.


In [103]:
retrieved_docs = retriever.invoke("Langchain")

In [104]:
print(retrieved_docs[0].page_content)

LangChain helps us to unlock the ability to harness the 
LLM’s immense potential in tasks such as document analysis, 
chatbot development, code analysis, and countless other 
applications. Whether your desire is to unlock deeper natural 
language understanding , enhance data, or circumvent 
language barriers through translation, LangChain is ready to 
provide the tools and programming support you need to do 
without it that it is not only difficult but also fresh for you. Its 
core functionalities encompass: 
1. Context-Aware Capabilities: LangChain facilitates the 
development of applications that are inherently 
context-aware. This means that these applications can 
connect to a language model and draw from various 
sources of context, such as prompt instructions, a few-
shot examples, or existing content, to ground their 
responses effectively. 
2. Reasoning Abilities: LangChain equips applications 
with the capacity to reason effectively. By relying on a 
language model, these appl

##### RetrievalQA


Now that you understand how to retrieve information from a document, you might be interested in exploring some more exciting applications. For instance, you could have the Language Model (LLM) read the paper and summarize it for you, or create a QA bot that can answer your questions based on the paper.

Here's an example using LangChain's `RetrievalQA`.


In [105]:
from langchain.chains import RetrievalQA

In [112]:
llm = HuggingFaceRouterLLM(model_id="zai-org/GLM-5.2")

In [113]:
qa = RetrievalQA.from_chain_type(llm=llm, 
                                 chain_type="stuff", 
                                 retriever=docsearch.as_retriever(), 
                                 return_source_documents=False)
query = "what is this paper discussing?"
qa.invoke(query)

{'query': 'what is this paper discussing?',
 'result': "I don't know. The provided context consists of fragmented sentences and lacks a coherent overarching topic to determine exactly what the paper is discussing."}

### Memory


Most LLM applications have a conversational interface. An essential component of a conversation is being able to refer to information introduced earlier in the conversation. At bare minimum, a conversational system should be able to access some window of past messages directly.


#### Chat message history


One of the core utility classes underpinning most (if not all) memory modules is the `ChatMessageHistory` class. This is a super lightweight wrapper that provides convenience methods for saving `HumanMessages`, `AIMessage`s, and then fetching them all.

Here is an example.


In [115]:
from langchain.memory import ChatMessageHistory

In [116]:
chat = llm_chat

history = ChatMessageHistory()

history.add_ai_message("hi!")

history.add_user_message("what is the capital of France?")

Let's have a look at the messages in the history:


In [117]:
history.messages

[AIMessage(content='hi!'),
 HumanMessage(content='what is the capital of France?')]

You can pass these messages in history to the model to generate a response:


In [118]:
ai_response = chat.invoke(history.messages)
ai_response

AIMessage(content='\n\nThe capital of France is **Paris**.', id='run-3dc6839a-257b-41fc-9e3e-c9368515e647-0')

You can see the model gives a proper response.


Let's have a look at the messages in the history again. Note that the history now includes the AI's message, which has been appended to the message history:


In [119]:
history.add_ai_message(ai_response)
history.messages

[AIMessage(content='hi!'),
 HumanMessage(content='what is the capital of France?'),
 AIMessage(content='\n\nThe capital of France is **Paris**.', id='run-3dc6839a-257b-41fc-9e3e-c9368515e647-0')]

#### Conversation buffer


This type of memory allows for the storage of messages, which can then be extracted to a variable. Consider using this in a chain, setting `verbose=True` so that the prompt can be visible.


In [120]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

In [121]:
conversation = ConversationChain(
    llm=llm,
    verbose=True,
    memory=ConversationBufferMemory()
)

Let’s begin the conversation by introducing the user as a little cat and proceed by incorporating some additional messages. Finally, prompt the model to check if it can recall that the user is a little cat.


In [122]:
conversation.invoke(input="Hello, I am a little cat. Who are you?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hello, I am a little cat. Who are you?
AI:

> Finished chain.


{'input': 'Hello, I am a little cat. Who are you?',
 'history': '',
 'response': "Hello there, little cat! *purrs softly in solidarity* I'm an AI assistant, a digital friend here to chat with you about anything you'd like. I don't have fur or whiskers like you do, but I'm always eager to learn and share what I know. Are you a tabby cat, a calico, or perhaps a sleek black panther? I'd love to hear more about you and what's on your mind today."}

In [123]:
conversation.invoke(input="What can you do?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hello, I am a little cat. Who are you?
AI: Hello there, little cat! *purrs softly in solidarity* I'm an AI assistant, a digital friend here to chat with you about anything you'd like. I don't have fur or whiskers like you do, but I'm always eager to learn and share what I know. Are you a tabby cat, a calico, or perhaps a sleek black panther? I'd love to hear more about you and what's on your mind today.
Human: What can you do?
AI:

> Finished chain.


{'input': 'What can you do?',
 'history': "Human: Hello, I am a little cat. Who are you?\nAI: Hello there, little cat! *purrs softly in solidarity* I'm an AI assistant, a digital friend here to chat with you about anything you'd like. I don't have fur or whiskers like you do, but I'm always eager to learn and share what I know. Are you a tabby cat, a calico, or perhaps a sleek black panther? I'd love to hear more about you and what's on your mind today.",
 'response': "Well, little cat, since I'm a digital creature, I don't have paws to chase mice or climb trees, but I do have quite a few tricks up my metaphorical sleeves! I can answer questions about history, science, and geography—like explaining why the ancient Egyptians revered your ancestors or how the solar system works. I can help with writing tasks, like composing a poem about a cat napping in a sunbeam or drafting a story about a feline adventurer. I can also brainstorm ideas, summarize information, or just chat about whatever

In [124]:
conversation.invoke(input="Who am I?.")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hello, I am a little cat. Who are you?
AI: Hello there, little cat! *purrs softly in solidarity* I'm an AI assistant, a digital friend here to chat with you about anything you'd like. I don't have fur or whiskers like you do, but I'm always eager to learn and share what I know. Are you a tabby cat, a calico, or perhaps a sleek black panther? I'd love to hear more about you and what's on your mind today.
Human: What can you do?
AI: Well, little cat, since I'm a digital creature, I don't have paws to chase mice or climb trees, but I do have quite a few tricks up my metaphorical sleeves! I can answer questions about history, science, and geography—like explaining w

{'input': 'Who am I?.',
 'history': "Human: Hello, I am a little cat. Who are you?\nAI: Hello there, little cat! *purrs softly in solidarity* I'm an AI assistant, a digital friend here to chat with you about anything you'd like. I don't have fur or whiskers like you do, but I'm always eager to learn and share what I know. Are you a tabby cat, a calico, or perhaps a sleek black panther? I'd love to hear more about you and what's on your mind today.\nHuman: What can you do?\nAI: Well, little cat, since I'm a digital creature, I don't have paws to chase mice or climb trees, but I do have quite a few tricks up my metaphorical sleeves! I can answer questions about history, science, and geography—like explaining why the ancient Egyptians revered your ancestors or how the solar system works. I can help with writing tasks, like composing a poem about a cat napping in a sunbeam or drafting a story about a feline adventurer. I can also brainstorm ideas, summarize information, or just chat about 

As you can see, the model remembers that the user is a little cat. You can see this in both the `history` and the `response` keys in the dictionary returned by the `conversation.invoke()` method.


### Chains


Chains refer to sequences of calls - whether to an LLM, a tool, or a data preprocessing step.

It combines different LLM calls and actions automatically.

Ex: Summary #1, Summary #2, Summary #3 > Final Summary


##### Simple LLMChain


Here is a simple single chain using `LLMChain`.


In [125]:
from langchain.chains import LLMChain

In [127]:
template = """Your job is to come up with a classic dish from the area that the users suggests.
                {location}
                
                YOUR RESPONSE:
"""
prompt_template = PromptTemplate(template=template, input_variables=['location'])

# chain 1
location_chain = LLMChain(llm=llm, prompt=prompt_template, output_key='meal')

In [128]:
location_chain.invoke(input={'location':'China'})

{'location': 'China',
 'meal': 'A classic dish from China is **Peking Duck** (北京烤鸭).\n\nOriginating from Beijing (historically known as Peking), this world-renowned dish dates back to the imperial era of the Ming Dynasty. It is famous for its thin, crispy skin and tender, succulent meat. \n\nTraditionally, the duck is roasted in a closed or hung oven over wood fire, which gives the meat a distinct fruity, smoky aroma. It is typically served in three courses: first, the crispy skin is served with sugar and sweet bean sauce; second, the meat is served with thin pancakes, scallions, and cucumber; and finally, the remaining bones are often used to make a rich, savory broth.'}

##### Simple sequential chain


Sequential chains allow the output of one LLM to be used as the input for another. This approach is beneficial for dividing tasks and maintaining the focus of your LLM.


In [129]:
from langchain.chains import SequentialChain

In [130]:
template = """Given a meal {meal}, give a short and simple recipe on how to make that dish at home.

                YOUR RESPONSE:
"""
prompt_template = PromptTemplate(template=template, input_variables=['meal'])

# chain 2
dish_chain = LLMChain(llm=llm, prompt=prompt_template, output_key='recipe')

In [131]:
template = """Given the recipe {recipe}, estimate how much time I need to cook it.

                YOUR RESPONSE:
"""
prompt_template = PromptTemplate(template=template, input_variables=['recipe'])

# chain 3
recipe_chain = LLMChain(llm=llm, prompt=prompt_template, output_key='time')

In [132]:
# overall chain
overall_chain = SequentialChain(chains=[location_chain, dish_chain, recipe_chain],
                                      input_variables=['location'],
                                      output_variables=['meal', 'recipe', 'time'],
                                      verbose= True)

In [133]:
from pprint import pprint

Let's use ```pprint``` to print the response to make it more clear.


In [134]:
pprint(overall_chain.invoke(input={'location':'China'}))



> Entering new SequentialChain chain...

> Finished chain.
{'location': 'China',
 'meal': 'A quintessential classic dish from China is **Peking Duck** (北京烤鸭).\n'
         '\n'
         'Originating from Beijing (historically known as Peking), this '
         'world-renowned dish has been a staple of Chinese imperial cuisine '
         'since the Ming Dynasty. \n'
         '\n'
         "**How it's made:**\n"
         'The preparation of Peking Duck is an elaborate, multi-step process. '
         'The duck is first pumped full of air to separate the skin from the '
         'fat, then glazed with a mixture of maltose syrup and roasted in a '
         'specialized, wood-fired oven (often using fruitwoods like apple or '
         'pear) until the skin turns a deep, glossy mahogany. \n'
         '\n'
         "**How it's served:**\n"
         'The duck is traditionally carved tableside by the chef. The prized, '
         'crispy skin and tender meat are served with thin, steamed pancakes

##### Summarization chain


Here is an example of using `load_summarize_chain` to summarize content.

Let's use the `web_data` that you loaded from LangChain before as the content that needs to be summarized.


In [135]:
from langchain.chains.summarize import load_summarize_chain

In [136]:
chain = load_summarize_chain(llm=llm, chain_type="stuff", verbose=False)
response = chain.invoke(web_data)

In [137]:
print(response['output_text'])

Here is a concise summary of the documentation:

LangChain provides `create_agent`, a minimal and highly configurable agent harness that lets you compose custom AI agents by combining a model, tools, prompts, and middleware. It offers a standard interface compatible with major AI providers (OpenAI, Anthropic, Google, etc.). 

The LangChain ecosystem is structured into three main tiers:
*   **LangChain (`create_agent`)**: For building highly customizable, tailored agent harnesses.
*   **Deep Agents**: A "batteries-included" solution built on LangChain for advanced features like automatic context compression and subagent spawning.
*   **LangGraph**: A low-level orchestration framework for complex workflows combining deterministic and agentic processes.

Additionally, **LangSmith** integrates with these frameworks to trace, debug, and evaluate agent behavior in production.


### Agents


##### Tools


Tools are interfaces that an agent, a chain, or a chat model / LLM can use to interact with the world.


You can find a list of tools that LangChain supports at [https://docs.langchain.com/oss/python/integrations/tools](https://docs.langchain.com/oss/python/integrations/tools).


Let’s explore how to work with tools, using the `Python REPL` tool as an example. The `Python REPL` tool can execute Python commands. These commands can either come from the user or be generated by the LLM. This tool is particularly useful for complex calculations. Instead of having the LLM generate the answer directly, it can be more efficient to have the LLM generate code to calculate the answer.


In [142]:
from langchain.agents import Tool
from langchain_experimental.utilities import PythonREPL

In [143]:
python_repl = PythonREPL()

Let's pass a simple Python command here as the input to let the tool excute.


In [144]:
python_repl.run("a = 3; b = 1; print(a+b)")

Python REPL can execute arbitrary code. Use with caution.


'4\n'

##### Toolkits


Toolkits are collections of tools that are designed to be used together for specific tasks.

Let's create a toolkit that contains one tool which is `PythonREPLTool`. Note that tools are put into a `list` object.


In [145]:
from langchain_experimental.tools import PythonREPLTool

In [146]:
tools = [PythonREPLTool()]

A list of toolkits that Langchain supports is available at [https://docs.langchain.com/oss/python/integrations/tools#all-tools-and-toolkits](https://docs.langchain.com/oss/python/integrations/tools#all-tools-and-toolkits).


##### Agents


By themselves, language models can't take actions - they just output text. A big use case for LangChain is creating agents. Agents are systems that use an LLM as a reasoning engineer to determine which actions to take and what the inputs to those actions should be. The results of those actions can then be fed back into the agent. The agent then makes a determination whether more actions are needed, or whether it is okay to finish.


Here you are going to create an agent that causes the LLM to generate Python code according to a coding question description.


In [147]:
from langchain.agents import create_react_agent
from langchain import hub
from langchain.agents import AgentExecutor

In [148]:
instructions = """You are an agent designed to write and execute python code to answer questions.
You have access to a python REPL, which you can use to execute python code.
If you get an error, debug your code and try again.
Only use the output of your code to answer the question. 
You might know the answer without running any code, but you should still run the code to get the answer.
If it does not seem like you can write code to answer the question, just return "I don't know" as the answer.
"""

# here you will use the prompt directly from the langchain hub
base_prompt = hub.pull("langchain-ai/react-agent-template")
prompt = base_prompt.partial(instructions=instructions)

In [149]:
base_prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'instructions', 'tool_names', 'tools'], partial_variables={'chat_history': ''}, metadata={'lc_hub_owner': 'langchain-ai', 'lc_hub_repo': 'react-agent-template', 'lc_hub_commit_hash': 'a246b05cc732c74d8852689863dc21ec21771bedd97b25eea767e0b22de31c76'}, template='{instructions}\n\nTOOLS:\n------\n\nYou have access to the following tools:\n\n{tools}\n\nTo use a tool, please use the following format:\n\n```\nThought: Do I need to use a tool? Yes\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n```\n\nWhen you have a response to say to the Human, or if you do not need to use a tool, you MUST use the format:\n\n```\nThought: Do I need to use a tool? No\nFinal Answer: [your response here]\n```\n\nBegin!\n\nPrevious conversation history:\n{chat_history}\n\nNew input: {input}\n{agent_scratchpad}')

You'll use the `create_react_agent` agent. It combines reasoning (e.g., Chain-of-Thought (CoT) prompting) and acting (e.g., action plan generation) together to let the LLM solve questions like humans would.

Now, set `verbose=True` to see how the LLM thinks and acts at every step.


In [150]:
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)  # tools were defined in the toolkit part above

Let's ask a coding question to solve LLM problem:


In [151]:
agent_executor.invoke(input = {"input": "What is the 3rd fibonacci number?"})



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Python_REPL
Action Input:
```python
def fibonacci(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

print(fibonacci(3))
```2
Do I need to use a tool? No
Final Answer: 2

> Finished chain.


{'input': 'What is the 3rd fibonacci number?', 'output': '2'}

# Exercises


### Exercise 1: Try with another LLM


Watsonx.ai provides access to several foundational models. In this lab, used `mistralai/mistral-small-3-1-24b-instruct-2503` has been used. Try using another foundational model, such as `meta-llama/llama-4-maverick-17b-128e-instruct-fp8`.


In [152]:
# Your code here

agent = create_react_agent(HuggingFaceRouterLLM(model_id="ibm-granite/granite-4.2-3b"), tools, prompt)
agent_executor.invoke(input = {"input": "What is the 3rd fibonacci number?"})



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Python_REPL
Action Input:
```python
def fibonacci(n):
    if n <= 0:
        return "Input must be a positive integer"
    elif n == 1:
        return 0
    elif n == 2:
        return 1
    else:
        a, b = 0, 1
        for _ in range(2, n):
            a, b = b, a + b
        return b

print(fibonacci(3))
```1
Do I need to use a tool? No
Final Answer: 1

> Finished chain.


{'input': 'What is the 3rd fibonacci number?', 'output': '1'}

<details>
    <summary>Click here for hint</summary>

```python
model_id = 'meta-llama/llama-4-maverick-17b-128e-instruct-fp8' 
```

</details>


<details>
    <summary>Click here for a hint about how to get the list of models</summary>

You can get a list of available models by putting in a random model name and getting the list of models from the error message:

```python
model_id = 'NONEXISTANT_MODEL_RANDOM_TEXT'

parameters = {
    GenParams.MAX_NEW_TOKENS: 256,  # this controls the maximum number of tokens in the generated output
    GenParams.TEMPERATURE: 0.5, # this randomness or creativity of the model's responses
}

credentials = {
    "url": "https://us-south.ml.cloud.ibm.com"
}

project_id = "skills-network"

model = ModelInference(
    model_id=model_id,
    params=parameters,
    credentials=credentials,
    project_id=project_id
)
```

</details>


<details>
    <summary>Click here for the solution</summary>

```python
model_id = 'meta-llama/llama-4-maverick-17b-128e-instruct-fp8' 

parameters = {
    GenParams.MAX_NEW_TOKENS: 256,  # this controls the maximum number of tokens in the generated output
    GenParams.TEMPERATURE: 0.5, # this randomness or creativity of the model's responses
}

credentials = {
    "url": "https://us-south.ml.cloud.ibm.com"
}

project_id = "skills-network"

model = ModelInference(
    model_id=model_id,
    params=parameters,
    credentials=credentials,
    project_id=project_id
)

llama_llm = WatsonxLLM(model=model)
```

</details>


### Exercise 2: Split the document with another separator


Can you use another separator to split the document and see how types of chunks are created? For example, use `["\n\n", "\n", ". ", " ", ""]` as separators.


In [159]:
# Your code here
ex_text_splitter = RecursiveCharacterTextSplitter(separators=["\n\n", "\n", ". ", " ", ""],chunk_size = 50,chunk_overlap = 10)

ex_chunks  = ex_text_splitter.split_documents(document)

In [157]:
ex_chunks[1]

Document(metadata={'source': 'assets/langchain-paper.pdf', 'page': 0}, page_content="mental challenges. It's like shifting from struggling with \ndifficult thoughts to considering ending everything. This shift \nis observable in how people express themselves and \ninteract[2]. \nOne practical approach to addressing mental illness and \npreventing suicidal ideation is early identification. Recent \nadvancements in deep learning have facilitated the \ndevelopment of effective early detection methods [3]. A \nnotable trend in natural language processing (NLP)  involves \nthe use of contextualized pretrained language models  [4], \nwhich have garnered substantial attention for their \neffectiveness in various text processing tasks. \nThis paper delves into the application of these recent \nadvancements in pretrained contextualized large language \nmodels to introduce MindGuide, an innovative chatbot \ndesigned to function as a mental health assistant for \nindividuals in need of guidance a

<details>
    <summary>Click here for Solution</summary>

```python
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20, separators=["\n\n", "\n", ". ", " ", ""])
chunks = text_splitter.split_documents(document)
print(len(chunks))

print(chunks[5].page_content)
```

</details>


### Exercise 3: Create an agent to talk with CSV data


##### Imagine you have a CSV file that you would like an LLM to read and analyze for you. This way, you only need to ask the LLM, and it can return the answer to you. You can use this assets/student-mat.csv a sample CSV file.

In [161]:
# Your code here


from langchain.agents.agent_types import AgentType
from langchain_experimental.agents.agent_toolkits import create_csv_agent
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
import pandas as pd

df = pd.read_csv(
    "assets/student-mat.csv"
)

agent = create_pandas_dataframe_agent(
    llm,
    df,
    verbose=True,
    return_intermediate_steps=True,
    allow_dangerous_code=True
)

response = agent.invoke("How many rows in the dataframe?",handle_parsing_errors=True)

print(response['output'])




> Entering new AgentExecutor chain...
Thought: I need to find the number of rows in the dataframe `df`. I can use the `len()` function or `df.shape[0]` to get this.
Action: python_repl_ast
Action Input: len(df)395I now know the final answer
Final Answer: 395

> Finished chain.
395


<details>
    <summary>Click here for Solution</summary>

```python
from langchain.agents.agent_types import AgentType
from langchain_experimental.agents.agent_toolkits import create_csv_agent
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
import pandas as pd

df = pd.read_csv(
    "assets/student-mat.csv"
)

agent = create_pandas_dataframe_agent(
    llama_llm,
    df,
    verbose=True,
    return_intermediate_steps=True
)

response = agent.invoke("How many rows in the dataframe?",handle_parsing_errors=True)

print(response['output'])
```

</details>


## Authors


[Kang Wang](https://author.skills.network/instructors/kang_wang)

Kang Wang is a Data Scientist in IBM. He is also a PhD Candidate in the University of Waterloo.


## Other contributors


[Wojciech Fulmyk](https://author.skills.network/instructors/wojciech_fulmyk)

Wojciech "Victor" Fulmyk is a Data Scientist at IBM. He is also a PhD Candidate in Economics in the University of Calgary.

[Faranak Heidari](https://author.skills.network/instructors/faranak_heidari) is a Data Scientist at IBM and PhD candidate at the University of Toronto.

[Karan Goswami](https://author.skills.network/instructors/karan_goswami) is a Data Scientist at IBM and is pursuing his Masters at McMaster University.


© Copyright IBM Corporation. All rights reserved.
